# Annotated & Visual Edition

This version keeps the original prediction logic intact and adds **English explanations, architecture diagrams, and logic-oriented visualizations** so that the notebook can be understood as a complete system rather than as isolated code cells.

## How to read this notebook

The pipeline has five conceptual layers:

1. **Surface-style models**  
   TF-IDF and stylometry estimate author identity from wording, punctuation, structure, and function-word patterns.

2. **Frozen LLM feature models**  
   Hidden states from Qwen3.5-9B and Gemma4-12B are mean-pooled at selected layers and classified by pre-fitted linear heads.

3. **LoRA author model**  
   A Qwen3-8B LoRA checkpoint provides another strong author-discrimination signal.

4. **Recovered topic groups**  
   A Qwen embedding model identifies passages that probably answer the same prompt. Group-centred heads reduce topic information shared by those passages.

5. **Conditional author decoding**  
   Within a recovered group, the final decoder computes exact marginal probabilities under the hypothesis that different passages in the same prompt group were produced by different authors.

The key distinction is:

> **The ensemble estimates author probabilities independently first; the exact decoder then conditions those probabilities on a group-level structural assumption.**

The original executable inference code is preserved below. The added cells are presentation and explanation only.


In [ ]:
# Visual overview: end-to-end inference architecture
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(15, 7))
ax.set_xlim(0, 15)
ax.set_ylim(0, 8)
ax.axis("off")

def box(x, y, w, h, title, subtitle=""):
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=1.5, facecolor="white"
    )
    ax.add_patch(patch)
    ax.text(x+w/2, y+h*0.62, title, ha="center", va="center",
            fontsize=11, fontweight="bold")
    if subtitle:
        ax.text(x+w/2, y+h*0.30, subtitle, ha="center", va="center",
                fontsize=8, wrap=True)

def arrow(x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle="-|>", mutation_scale=14, linewidth=1.2
    ))

box(0.3, 3.1, 1.7, 1.3, "Test passages", "id + raw text")

box(2.5, 6.0, 2.1, 1.2, "TF-IDF", "4 text views")
box(2.5, 4.2, 2.1, 1.2, "Stylometry", "LR + LightGBM")
box(2.5, 2.4, 2.1, 1.2, "Frozen LLMs", "Qwen3.5 + Gemma4")
box(2.5, 0.6, 2.1, 1.2, "LoRA model", "Qwen3-8B")

box(5.1, 2.7, 2.3, 1.8, "Log-probability\nensemble",
    "Weighted log-linear blend")

box(5.1, 5.5, 2.3, 1.4, "Topic embedding",
    "Qwen3-Embedding-0.6B")
box(8.0, 5.5, 2.3, 1.4, "Recovered groups",
    "cosine > 0.92\ncap size = 6")

box(8.0, 2.7, 2.3, 1.8, "Group-centred\nadjustment",
    "remove shared topic signal")

box(10.9, 2.7, 2.4, 1.8, "Exact marginal\ndecoder",
    "injective author assignments\nwithin each group")

box(13.7, 3.1, 1.0, 1.3, "Final\nlabel", "18 classes")

for yy in [6.6, 4.8, 3.0, 1.2]:
    arrow(2.0, 3.75, 2.45, yy)

for yy in [6.6, 4.8, 3.0, 1.2]:
    arrow(4.6, yy, 5.05, 3.6)

arrow(2.0, 3.75, 5.05, 6.2)
arrow(7.4, 6.2, 7.95, 6.2)
arrow(7.4, 3.6, 7.95, 3.6)
arrow(9.15, 5.5, 9.15, 4.55)
arrow(10.3, 3.6, 10.85, 3.6)
arrow(13.3, 3.6, 13.65, 3.6)

ax.text(7.5, 7.6, "Name That LLM — Full Inference Flow", ha="center",
        fontsize=15, fontweight="bold")
plt.show()


# Conditional Author Decoding for Name That LLM

**Public score: 0.520 (version 5). Offline inference on 2× T4.**

This notebook explores a small structural change to an LLM attribution ensemble: classify related passages jointly instead of treating their predictions as independent.

It builds on [Ahmed Berat Ozer's `name-that-llm-d` v1](https://www.kaggle.com/code/ahmedberatozer/name-that-llm-d?scriptVersionId=358776684), whose public score is **0.522**. Our modification **did not improve that leaderboard score**. The value of the experiment is the exact decoder, its validation protocol, and a reproducible account of what transferred and what did not.

## What changes?

The backbone ensemble stays the same: frozen Qwen3.5-9B and Gemma4-12B multilayer readouts, a Qwen3-8B LoRA checkpoint, four TF-IDF views, and stylometry classifiers. Topic embeddings recover small groups of passages answering similar questions. The original pipeline already has group-centred heads that reduce shared topic information.

The final heuristic subtracts some of the other passages' author logits. Here we replace it with **exact marginal probabilities over distinct-author assignments within a recovered group**. There is no constraint on global author frequencies.

## Run it

1. Copy the notebook with its attached inputs.
2. Select **GPU T4 ×2** and keep **Internet off**.
3. Use **Save & Run All**, then submit that saved version's `submission.parquet`.

The visible test file has three demonstration rows. Kaggle substitutes the hidden test set during submission. Version 5 is the scored version; subsequent presentation-only changes preserve its prediction code. The exported head arrays are fitted on the official training set after experimental choices were fixed. Their manifest binds them to the train file, label order, and backbone layers.

**Public fitted heads:** [NTL Frozen Heads and CE Projection](https://www.kaggle.com/datasets/kirilltym/ntl-frozen-heads-and-ce-projection). The fitted arrays are available under Apache 2.0; no private training-feature cache is needed for inference.


## How to read this notebook

This notebook has two goals:

1. **Build a competitive author-attribution ensemble** from heterogeneous signals.
2. **Test whether local structural constraints can improve row-wise predictions** when several passages probably belong to the same hidden prompt/topic group.

The most important distinction is:

- **Base prediction quality** comes from TF-IDF, stylometry, frozen LLM probes, and the LoRA model.
- **Structural post-processing** comes from topic-group recovery, group-centred features, and exact conditional decoding.

The structural layer can only help when the recovered groups are sufficiently accurate. If unrelated passages are grouped together, the same mechanism can hurt.

> **Potential score improvement:** the most promising gains are likely to come from improving the *quality of the base probabilities* and the *precision of recovered groups*, rather than making the exact decoder itself more complicated.

In [ ]:
# Roadmap: where information enters the pipeline
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 16)
ax.set_ylim(0, 9)
ax.axis("off")

def add_box(x, y, w, h, title, subtitle=""):
    p = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.04,rounding_size=0.08",
        linewidth=1.4, facecolor="white"
    )
    ax.add_patch(p)
    ax.text(x+w/2, y+h*0.64, title, ha="center", va="center",
            fontsize=10.5, fontweight="bold")
    if subtitle:
        ax.text(x+w/2, y+h*0.31, subtitle, ha="center", va="center",
                fontsize=8.2, wrap=True)

def add_arrow(x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle="-|>", mutation_scale=13, linewidth=1.2
    ))

add_box(0.4, 6.9, 2.2, 1.1, "Input passages", "test text")
add_box(3.4, 7.0, 2.4, 1.0, "TF-IDF", "surface wording / formatting")
add_box(3.4, 5.5, 2.4, 1.0, "Stylometry", "hand-crafted style statistics")
add_box(3.4, 4.0, 2.4, 1.0, "Frozen LLM probes", "Qwen3.5 + Gemma4 hidden states")
add_box(3.4, 2.5, 2.4, 1.0, "Qwen3-8B LoRA", "task-specialized neural signal")
add_box(3.4, 0.9, 2.4, 1.0, "Topic embedding", "recover likely prompt groups")

add_box(7.0, 5.2, 2.5, 1.2, "Component probabilities", "18-author distributions")
add_box(7.0, 3.2, 2.5, 1.2, "Group-centred views", "remove shared topic signal")
add_box(10.5, 5.2, 2.2, 1.2, "Weighted log blend", "Σ w · log p")
add_box(10.5, 3.2, 2.2, 1.2, "Group correction", "plain ↔ centred routing")
add_box(13.4, 4.2, 2.1, 1.3, "Exact marginals", "distinct-author prior")
add_box(13.4, 2.0, 2.1, 1.1, "Final label", "row-wise argmax")

for yy in [7.5, 6.0, 4.5, 3.0]:
    add_arrow(2.6, 7.45, 3.35, yy)
add_arrow(2.6, 7.45, 3.35, 1.4)

for yy in [7.5, 6.0, 4.5, 3.0]:
    add_arrow(5.85, yy, 6.95, 5.8)
add_arrow(5.85, 1.4, 6.95, 3.8)

add_arrow(9.55, 5.8, 10.45, 5.8)
add_arrow(9.55, 3.8, 10.45, 3.8)
add_arrow(12.75, 5.8, 13.35, 4.9)
add_arrow(12.75, 3.8, 13.35, 4.7)
add_arrow(14.45, 4.15, 14.45, 3.15)

ax.set_title(
    "End-to-End Architecture: Independent Evidence + Recovered Group Structure",
    fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.show()

## 1. Exact marginals, not a hard matching

Let $p_{ic}$ be the independent probability that passage $i$ was written by author $c$. For a group of $g$ passages, consider injective assignments $a$ (different authors for different passages):

$$P(a\mid\text{distinct}) = \frac{\prod_{i=1}^{g} p_{i,a_i}}{\sum_{b\ \mathrm{injective}} \prod_{i=1}^{g}p_{i,b_i}}.$$

The decoder sums this distribution to obtain each passage's marginal $q_{ic}=P(a_i=c\mid\text{distinct})$. We predict the largest marginal for each row. This is different from selecting the single most probable joint assignment: row-wise marginal argmax can still repeat an author.

The prior comes from the official training set, where each author occurs at most once per prompt. Its validity on hidden test groups is a hypothesis. Incorrect topic grouping can hurt. Singletons are left unchanged, oversized groups are skipped, and no uniform test class counts are imposed.

A dynamic program over subsets of passages evaluates the marginals in $O(Kg2^g)$ time. Here $K=18$ and $g\leq6$. The implementation was checked against exhaustive enumeration on a small case, and for permutation and singleton invariance.

### Logic note — why exact marginals matter

The function `exclusive_marginals()` does **not** simply choose one globally optimal author assignment.

Instead, it considers every valid injective assignment inside a group, weights each assignment by the product of the row-wise probabilities, and then **sums over all assignments that place author `c` on passage `i`**.

That produces a new matrix:

\[
q_{ic}=P(a_i=c\mid \text{all authors in the group are distinct})
\]

This is important because marginal uncertainty is preserved. A hard Hungarian-style assignment would collapse the whole group to one permutation, while this decoder keeps multiple plausible joint explanations alive.


In [ ]:
# Visual intuition: independent probabilities vs. conditional marginals
import numpy as np
import matplotlib.pyplot as plt

toy = np.array([
    [0.50, 0.35, 0.10],
    [0.50, 0.50, 0.10],
    [0.15, 0.25, 0.50],
], dtype=float)

def _exclusive_marginals_demo(p):
    from itertools import permutations
    n, k = p.shape
    assignments = []
    weights = []
    for perm in permutations(range(k), n):
        w = np.prod([p[i, perm[i]] for i in range(n)])
        assignments.append(perm)
        weights.append(w)
    weights = np.asarray(weights)
    weights /= weights.sum()
    q = np.zeros_like(p)
    for perm, w in zip(assignments, weights):
        for i, c in enumerate(perm):
            q[i, c] += w
    return q

toy_q = _exclusive_marginals_demo(toy)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, mat, title in zip(
    axes,
    [toy, toy_q],
    ["Independent probabilities", "Conditioned on distinct authors"]
):
    im = ax.imshow(mat, vmin=0, vmax=1, aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("Author class")
    ax.set_ylabel("Passage")
    ax.set_xticks(range(mat.shape[1]))
    ax.set_yticks(range(mat.shape[0]))
    for r in range(mat.shape[0]):
        for c in range(mat.shape[1]):
            ax.text(c, r, f"{mat[r,c]:.2f}", ha="center", va="center", fontsize=9)

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8, label="Probability")
fig.suptitle("Exact decoding redistributes probability across a whole group", y=1.03)
plt.show()


In [ ]:
# NTL conditional author decoding experiment, 2026-10-16.
# Base: Ahmed Berat Ozer, name-that-llm-d v1, Kaggle Apache-2.0, public .522.
# https://www.kaggle.com/code/ahmedberatozer/name-that-llm-d?scriptVersionId=358776684
# Experimental change: exact marginal decoding within inferred topic groups.
# No global class-count constraint; no validation labels used at inference.
import numpy as np
from scipy.special import softmax
def exclusive_marginals(probabilities):
    p = np.asarray(probabilities, dtype=np.float64)
    if p.ndim != 2 or not np.isfinite(p).all() or (p < 0).any() or (p.sum(1) <= 0).any():
        raise ValueError('Expected finite nonnegative probability rows')
    p = p / p.sum(1, keepdims=True)
    n, k = p.shape
    if n > k or n > 8:
        raise ValueError('Exact decoding supports at most 8 passages and at least as many classes')
    if n <= 1:
        return p.copy()
    full = (1 << n) - 1
    f = np.zeros((k + 1, 1 << n))
    b = np.zeros_like(f)
    f[0, 0] = 1.0
    b[k, 0] = 1.0
    for c in range(k):
        f[c + 1] = f[c]
        for mask in range(1, full + 1):
            for i in range(n):
                if mask & 1 << i:
                    f[c + 1, mask] += p[i, c] * f[c, mask ^ 1 << i]
    for c in range(k - 1, -1, -1):
        b[c] = b[c + 1]
        for mask in range(1, full + 1):
            for i in range(n):
                if mask & 1 << i:
                    b[c, mask] += p[i, c] * b[c + 1, mask ^ 1 << i]
    z = f[k, full]
    if z <= np.finfo(float).tiny:
        return p.copy()
    q = np.zeros_like(p)
    for i in range(n):
        remaining = full ^ 1 << i
        for c in range(k):
            subset = remaining
            while True:
                q[i, c] += p[i, c] * f[c, subset] * b[c + 1, remaining ^ subset] / z
                if subset == 0:
                    break
                subset = subset - 1 & remaining
    return q / q.sum(1, keepdims=True)

def repulsion(logits, groups, alpha=0.2):
    z = np.asarray(logits, dtype=float).copy()
    for group in np.unique(groups):
        rows = np.flatnonzero(groups == group)
        if len(rows) > 1:
            z[rows] -= alpha * (logits[rows].sum(0) - logits[rows]) / (len(rows) - 1)
    return z

def decode(logits, groups, strength=1.0, confidence=None, max_group=6, temperature=1.0):
    p = softmax(np.asarray(logits, dtype=float) / temperature, axis=1)
    q = p.copy()
    for group in np.unique(groups):
        rows = np.flatnonzero(groups == group)
        if not 1 < len(rows) <= max_group:
            continue
        weight = float(strength if confidence is None else strength * confidence[group])
        if not 0 <= weight <= 1:
            raise ValueError('Mixture weight outside [0,1]')
        q[rows] = (1 - weight) * p[rows] + weight * exclusive_marginals(p[rows])
    return np.log(q.clip(1e-300))


def fit_ce_predict(xtrain, y, prompts, xtest, log=print, device=None, epochs=20):
    import numpy as np
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from sklearn.preprocessing import StandardScaler
    from scipy.special import log_softmax
    torch.set_num_threads(6)
    torch.manual_seed(42)
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
    scaler = StandardScaler().fit(xtrain)
    z = np.clip(scaler.transform(xtrain), -8, 8).astype(np.float32)
    rng = np.random.default_rng(42)
    unique = np.unique(prompts)
    k = len(np.unique(y))

    class Net(nn.Module):

        def __init__(self):
            super().__init__()
            self.proj = nn.Linear(z.shape[1], 128)
            self.drop = nn.Dropout(0.2)
            self.plain = nn.Linear(128, k)
            self.center = nn.Linear(128, k)

        def forward(self, a):
            h = torch.tanh(self.proj(self.drop(a)))
            return (h, self.plain(h), self.center(h))
    net = Net().to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=0.0005, weight_decay=0.1)
    for epoch in range(epochs):
        ordered = rng.permutation(unique)
        for start in range(0, len(unique), 8):
            ix = np.flatnonzero(np.isin(prompts, ordered[start:start + 8]))
            a = torch.as_tensor(z[ix], device=device)
            target = torch.as_tensor(y[ix], dtype=torch.long, device=device)
            _, plain, _ = net(a)
            loss = F.cross_entropy(plain, target, label_smoothing=0.05)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
        if (epoch + 1) % 5 == 0:
            log(f'CE style projection: epoch {epoch + 1}/{epochs}')
    net.eval()
    weights = {key: val.detach().cpu().numpy() for key, val in net.state_dict().items()}
    ztest = np.clip(scaler.transform(xtest), -8, 8).astype(np.float32)
    h = np.tanh(ztest @ weights['proj.weight'].T + weights['proj.bias'])
    logits = h @ weights['plain.weight'].T + weights['plain.bias']
    del net, opt
    if device == 'cuda':
        torch.cuda.empty_cache()
    return log_softmax(logits, axis=1)


def apply_scaler(x, mean, scale):
    import numpy as np
    z = np.asarray(x, dtype=np.float32).copy()
    z -= mean
    z /= scale
    return z

def linear_logproba(x, coef, bias):
    from scipy.special import log_softmax
    return log_softmax(x @ coef.T + bias, axis=1)

def prefit_ce_predict(x, weights):
    import numpy as np
    from scipy.special import log_softmax
    z = apply_scaler(x, weights['ce_mean'], weights['ce_scale'])
    z = np.clip(z, -8, 8)
    h = np.tanh(z @ weights['ce_proj_weight'].T + weights['ce_proj_bias'])
    return log_softmax(h @ weights['ce_plain_weight'].T + weights['ce_plain_bias'], axis=1)

### Decoder complexity and practical limits

For a group of size \(g\) and \(K\) authors, exact enumeration considers

\[
P(K,g)=\frac{K!}{(K-g)!}
\]

injective assignments.

With 18 authors, this grows quickly:

| Group size | Number of injective assignments |
|---:|---:|
| 2 | 306 |
| 3 | 4,896 |
| 4 | 73,440 |
| 5 | 1,028,160 |
| 6 | 13,366,080 |

This explains the conservative `GROUP_CAP = 6`. The decoder is exact, but exactness becomes expensive very quickly.

> **Potential score improvement:** rather than increasing `GROUP_CAP`, it is more attractive to improve group precision and confidence-gate the decoder. A wrong six-row group can damage more predictions than a missed group.

## 2. Environment and official data

The same sorted list of 18 authors is used by every branch. We verify the train and exported-weight hashes before prediction. Text and stylometry classifiers are fitted from the official train; no external response corpus is used in the final model.

### Configuration map

The configuration block defines the experiment's **fixed inference contract**:

- `LABELS`: the 18 possible LLM authors.
- `PROBES`: selected hidden layers from Gemma4-12B and Qwen3.5-9B.
- `LORAS`: the Qwen3-8B LoRA checkpoint.
- `WEIGHTS`: the log-linear ensemble weights.
- `GROUP_THR = 0.92`: cosine-similarity threshold for topic linking.
- `GROUP_CAP = 6`: exact decoding is only attempted for small groups.
- `GROUP_WS`: how strongly each neural component moves toward its group-centred head.

The design intentionally separates **model evidence** from **group structure**. First each model produces class evidence; then group information modifies that evidence.


In [ ]:
# Visualize the ensemble weights used by the original notebook
import matplotlib.pyplot as plt

weights_demo = {
    'Qwen3.5 frozen': 0.326,
    'Gemma4 frozen': 0.125,
    'Qwen3-8B LoRA': 0.413,
    'TF-IDF': 0.249,
    'Stylometry LR': 0.040,
    'Stylometry LGBM': 0.081,
}

names = list(weights_demo)
vals = list(weights_demo.values())

fig, ax = plt.subplots(figsize=(9, 4.2))
bars = ax.barh(names, vals)
ax.set_xlabel("Log-linear blend weight")
ax.set_title("Relative contribution of ensemble components")
ax.invert_yaxis()
for b, v in zip(bars, vals):
    ax.text(v + 0.006, b.get_y() + b.get_height()/2, f"{v:.3f}",
            va="center", fontsize=9)
plt.tight_layout()
plt.show()


### Interpreting ensemble weights

The numeric weights are not direct percentages. They scale **log-probability evidence**, so a branch with a smaller weight may still decisively affect difficult rows when its distribution is much sharper than the others.

A useful mental model is:

| Component | What it is good at | Typical failure mode | Improvement potential |
|---|---|---|---|
| TF-IDF | lexical habits, punctuation, phrase reuse | topic leakage | Medium |
| Stylometry LR | stable low-dimensional style cues | underfitting nonlinear interactions | Low–Medium |
| Stylometry LGBM | nonlinear style interactions | overfitting / unstable calibration | Medium |
| Frozen Qwen3.5 | latent stylistic representation | truncation / pooling mismatch | High |
| Frozen Gemma4 | complementary latent representation | correlated errors with Qwen | Medium–High |
| Qwen3-8B LoRA | task-specific author separation | calibration / overconfidence | High |

> **Potential score improvement:** recalibrating individual component probabilities before blending, and re-fitting blend weights on strict prompt-held-out OOF predictions, is one of the safest places to search for leaderboard gains.

In [ ]:
"""Name That LLM - full-fit + inference pipeline (Kaggle T4x2 offline, or local validation).

Components (each -> (n_test, 18) log-probs), blended log-linearly with weights fitted on local OOF:
  * tfidf   : style-aware 4-view TF-IDF (char 3-5, word 1-3, shape-text, function-word text) + LR      [CPU]
  * stylo   : hand-crafted stylometry -> LR and LightGBM                                              [CPU]
  * probes  : frozen LM hidden states (selected early/mid layers, mean-pooled w/o first token)
              -> per-layer StandardScaler+LR, log-probs averaged over layers                           [GPU]
  * lora    : LoRA-fine-tuned Qwen3-8B (first 10 layers, multi-layer pooled head)                       [GPU]
Optional test-time grouping (USE_GROUPS): passages linked by topic-embedding similarity get extra
predictions from "group-centred" probe heads (topic removed), blended in with weight GROUP_W.

Local validation:  python ntl_infer.py --local_fold 0   (fit on folds != 0, predict fold 0, report bacc)
"""
import os, sys, re, gc, glob, json, time, argparse
from pathlib import Path
import numpy as np, pandas as pd

ap = argparse.ArgumentParser()
ap.add_argument('--local_fold', type=int, default=-1)
ap.add_argument('--skip_gpu', action='store_true')
args, _ = ap.parse_known_args()
T0 = time.time()
def log(m): print(f'[{time.time() - T0:7.1f}s] {m}', flush=True)

HERE = Path(__file__).resolve().parent if '__file__' in globals() else Path('.')
ON_KAGGLE = Path('/kaggle/input').exists()
sys.path.insert(0, str(HERE))
if ON_KAGGLE:
    # code dataset is flat: stylo.py, lora_infer.py and ckpt__<name>__<file> -> rebuild ckpt dirs in /kaggle/working
    import shutil
    code_dir = Path(sorted(glob.glob('/kaggle/input/**/lora_infer.py', recursive=True))[0]).parent
    sys.path.insert(0, str(code_dir))
    for f in code_dir.glob('ckpt__*__*'):
        _, cname, fname = f.name.split('__', 2)
        dst = Path('/kaggle/working/ckpt') / cname
        dst.mkdir(parents=True, exist_ok=True)
        shutil.copy(f, dst / fname)
    log(f'code dir {code_dir}; ckpts {sorted(p.name for p in Path("/kaggle/working/ckpt").glob("*"))}')
import torch, sklearn, transformers
from threadpoolctl import threadpool_limits
_CPU_POOL_LIMIT = threadpool_limits(limits=4)
log(f'versions: torch {torch.__version__} transformers {transformers.__version__} sklearn {sklearn.__version__} '
    f'numpy {np.__version__} pandas {pd.__version__} | gpus {torch.cuda.device_count()} '
    f'{[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}')

# ------------------------------------------------------------------ config
USE_GROUPS = os.environ.get('NTL_GROUPS', '1') == '1'
GROUP_THR, GROUP_W, GROUP_CAP, LOGIT_ALPHA = 0.92, 1.0, 6, 0.2
GROUP_WS = {'q35': 1.0, 'g12': 1.0, 'q8lora': 0.5}  # centred-vs-plain mix per component (LoRA: fold0 50/50 best)
TEST_STD = os.environ.get('NTL_TEST_STD', '1') == '1'
PROBES = [
    dict(name='g12', match=dict(hidden_size=3840, num_hidden_layers=48), layers=[2, 4, 6, 8, 10, 12, 14, 16, 20, 24, 28], C=0.002),
    dict(name='q35', match=dict(hidden_size=4096, num_hidden_layers=32), layers=[2, 3, 4, 5, 6, 17, 18, 19], C=0.002),
]
LORAS = [dict(name='q8lora', base_match=dict(model_type='qwen3', hidden_size=4096, num_hidden_layers=36),
              ckpts=['q8c_L24_s42_full'])]
# log-linear blend weights fitted on 5-fold OOF (v C: 24L LoRA + test-std probes; nested CV bacc 0.4978, NLL 1.376)
WEIGHTS = {'q35': 0.326, 'g12': 0.125, 'q8lora': 0.413, 'tfidf': 0.249, 'stylo_lr': 0.040, 'stylo_lgb': 0.081}
LOCAL_MODELS = {'g12': 'gemma-4-12B', 'q35': 'Qwen3.5-9B', 'q8lora': 'Qwen3-8B-Base', 'emb': 'Qwen3-Embedding-0.6B'}

LABELS = np.array(sorted(['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
                          'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
                          'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
                          'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']))
K = len(LABELS)
L2I = {l: i for i, l in enumerate(LABELS)}

# ------------------------------------------------------------------ data
if ON_KAGGLE:
    DATA = Path(sorted(glob.glob('/kaggle/input/**/name-that-llm/train.csv', recursive=True))[0]).parent
    OUT = Path('/kaggle/working')
    MODEL_ROOTS, CKPT_ROOTS = ['/kaggle/input'], ['/kaggle/working/ckpt']
else:
    ROOT = HERE.parent
    DATA, OUT = ROOT, ROOT / 'kaggle' / 'out'
    MODEL_ROOTS, CKPT_ROOTS = [str(ROOT / 'models')], [str(ROOT / 'ckpt')]
OUT.mkdir(parents=True, exist_ok=True)
train = pd.read_csv(DATA / 'train.csv', dtype=str)
test = pd.read_csv(DATA / 'test.csv', dtype=str)
test['text'] = test['text'].fillna('')
y_hold = None
if args.local_fold >= 0:
    from sklearn.model_selection import GroupKFold
    fid = np.zeros(len(train), dtype=int)
    for i, (_, v) in enumerate(GroupKFold(5).split(train, groups=train.prompt_id)):
        fid[v] = i
    test = train[fid == args.local_fold][['id', 'text']].reset_index(drop=True)
    y_hold = train.model.map(L2I).to_numpy()[fid == args.local_fold]
    train = train[fid != args.local_fold].reset_index(drop=True)
y = train.model.map(L2I).to_numpy()
groups_tr = train.prompt_id.to_numpy()
tr_texts, te_texts = train.text.tolist(), test.text.tolist()
log(f'train {len(train)} test {len(test)} kaggle={ON_KAGGLE} groups={USE_GROUPS}')


def logp_norm(lp):
    lp = lp - lp.max(1, keepdims=True)
    return lp - np.log(np.exp(lp).sum(1, keepdims=True))


COMP = {}
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack

## 3. Surface features

Four TF-IDF views capture character n-grams, word n-grams, word-shape patterns, and function-word patterns. Hand-crafted stylometry contributes a linear and a LightGBM readout. These branches provide information complementary to hidden-state features.

### Surface-feature branch: what information is being captured?

The four TF-IDF views deliberately look at the same answer through different lenses:

| View | Main signal |
|---|---|
| Character 3–5 grams | punctuation, spelling, formatting, token fragments |
| Word 1–3 grams | recurring wording and short phrases |
| Shape text | capitalization and orthographic structure |
| Function-word text | grammatical habits while suppressing topic vocabulary |

Stylometry adds hand-engineered statistics. Together, these branches are useful because they can detect author-specific habits that may not be cleanly separated in semantic LLM embeddings.


In [ ]:
# ------------------------------------------------------------------ classical
FUNCTION_WORDS = set(('a about above after again against all am an and any are as at be because been before being below between both '
                      'but by can could did do does doing down during each few for from further had has have having he her here hers '
                      'herself him himself his how i if in into is it its itself just me more most my myself no nor not now of off on '
                      'once only or other our ours ourselves out over own same she should so some such than that the their theirs them '
                      'themselves then there these they this those through to too under until up very was we were what when where which '
                      'while who whom why will with would you your yours yourself yourselves yet even still never always every rather '
                      'perhaps almost quite simply truly deeply instead whether though although thus hence therefore however').split())
shape_text = lambda t: re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)
function_text = lambda t: " ".join(w.lower() if w.lower() in FUNCTION_WORDS or not w.isalnum() else "CONTENT" for w in re.findall(r"\w+|[^\w\s]", t))
VIEWS = [((lambda t: t), dict(analyzer='char', ngram_range=(3, 5), max_features=250000), 1.0),
         ((lambda t: t), dict(analyzer='word', ngram_range=(1, 3), max_features=180000, token_pattern=r"(?u)\b\w+\b|[^\w\s]"), 0.7),
         (shape_text, dict(analyzer='char', ngram_range=(2, 6), max_features=60000), 0.5),
         (function_text, dict(analyzer='word', ngram_range=(1, 3), max_features=50000, token_pattern=r"\S+"), 0.5)]
A, B = [], []
for fn, opts, w in VIEWS:
    v = TfidfVectorizer(min_df=2, sublinear_tf=True, lowercase=False, **opts)
    A.append(v.fit_transform([fn(t) for t in tr_texts]) * w); B.append(v.transform([fn(t) for t in te_texts]) * w)
COMP['tfidf'] = LogisticRegression(C=50, max_iter=3000).fit(hstack(A).tocsr(), y).predict_log_proba(hstack(B).tocsr())
del A, B
log('tfidf done')

from stylo import stylo_matrix
Str, Ste = stylo_matrix(tr_texts), stylo_matrix(te_texts)
sc = StandardScaler().fit(Str)
COMP['stylo_lr'] = LogisticRegression(C=0.05, max_iter=5000).fit(sc.transform(Str), y).predict_log_proba(sc.transform(Ste))
import lightgbm as lgb
gbm = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=10, subsample=0.8,
                         subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0, verbose=-1).fit(Str, y)
COMP['stylo_lgb'] = np.log(np.clip(gbm.predict_proba(Ste), 1e-9, 1))
log('stylo done')


# ------------------------------------------------------------------ helpers for LMs

### Potential score improvement — surface-feature branch

Possible experiments, ordered from relatively safe to more invasive:

1. **Calibration per classifier**  
   Apply temperature scaling or isotonic calibration on prompt-held-out OOF predictions before the log blend.

2. **Author-aware feature pruning**  
   Remove n-grams that mainly encode topic rather than style. This can reduce prompt leakage.

3. **Character-range sweep**  
   Compare `(2,5)`, `(3,6)`, and `(4,7)` character n-grams. Different LLMs can leave different punctuation/token-fragment signatures.

4. **TF-IDF sublinear scaling / normalization sweep**  
   Small preprocessing changes can alter linear separability.

5. **Cross-feature interactions for stylometry**  
   Ratios such as punctuation-per-sentence, uppercase-per-token, or sentence-length dispersion can expose consistent generation habits.

These changes are plausible score-improvement candidates, but they must be validated with **prompt-level holdout**, not random row splits.

In [ ]:
# Surface-feature intuition map
import matplotlib.pyplot as plt

labels = [
    "Character n-grams",
    "Word n-grams",
    "Shape text",
    "Function words",
    "Stylometry"
]
style_signal = [5, 3, 4, 4, 5]
topic_risk = [2, 5, 2, 1, 1]

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(topic_risk, style_signal, s=[420, 420, 420, 420, 520])

for x, y, label in zip(topic_risk, style_signal, labels):
    ax.text(x + 0.06, y + 0.05, label, fontsize=9)

ax.set_xlabel("Relative topic-leakage risk")
ax.set_ylabel("Relative style sensitivity")
ax.set_title("Conceptual Role of Surface Features")
ax.set_xlim(0.5, 5.7)
ax.set_ylim(0.5, 5.7)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 4. Frozen model features

Mean-pool selected hidden layers, excluding the first token. Models are loaded sequentially. The extraction recipe, truncation, and test-statistics adaptation follow the referenced pipeline.

### Frozen hidden-state probes: data flow

For every selected transformer layer:

1. Tokenize the passage.
2. Run the frozen base model.
3. Mean-pool token hidden states, excluding the first token.
4. Standardize the feature vector.
5. Apply a pre-fitted linear classifier.
6. Average log-probabilities across selected layers.

The notebook also keeps a small subset of hidden-layer features for the compact CE projection ablation.

A key operational property is that the large backbones are **feature extractors only** here; their parameters are not updated during final inference.


In [ ]:
# Schematic of a frozen-layer probe
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 3)
ax.axis("off")

steps = [
    ("Passage", "raw text"),
    ("Tokenizer", "≤512 tokens"),
    ("Frozen LLM", "selected layers"),
    ("Mean pooling", "exclude token 0"),
    ("Scaler + LR", "per-layer head"),
    ("Log-probabilities", "18 authors"),
]
xs = [0.2, 2.1, 4.0, 6.0, 8.0, 10.1]

for x, (a, b) in zip(xs, steps):
    p = FancyBboxPatch((x, 1.0), 1.55, 1.0,
                       boxstyle="round,pad=0.03",
                       facecolor="white", linewidth=1.3)
    ax.add_patch(p)
    ax.text(x+0.775, 1.62, a, ha="center", va="center",
            fontsize=10, fontweight="bold")
    ax.text(x+0.775, 1.28, b, ha="center", va="center", fontsize=8)

for i in range(len(xs)-1):
    ax.add_patch(FancyArrowPatch((xs[i]+1.55,1.5),(xs[i+1]-0.08,1.5),
                                arrowstyle="-|>", mutation_scale=13))

ax.text(6, 2.55, "Frozen hidden-state author probe", ha="center",
        fontsize=14, fontweight="bold")
plt.show()


In [ ]:
def find_model_dir(match, local_name=None):
    if not ON_KAGGLE and local_name:
        d = Path(MODEL_ROOTS[0]) / local_name
        return str(d) if d.exists() else None
    for root in MODEL_ROOTS:
        for cfg in sorted(glob.glob(f'{root}/**/config.json', recursive=True)):
            try:
                c = json.load(open(cfg))
            except Exception:
                continue
            c = c.get('text_config', c)
            if all(c.get(k) == v for k, v in match.items()) and glob.glob(os.path.dirname(cfg) + '/*.safetensors'):
                return os.path.dirname(cfg)
    return None


def extract(model_dir, texts, layers, batch=8, max_len=512):
    import torch
    from transformers import AutoConfig, AutoModel, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_dir)
    tok.padding_side = 'right'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    cfg = AutoConfig.from_pretrained(model_dir)
    tcfg = cfg.get_text_config() if hasattr(cfg, 'get_text_config') else cfg
    tcfg.num_hidden_layers = max(layers)
    if getattr(tcfg, 'layer_types', None) is not None:
        tcfg.layer_types = tcfg.layer_types[:max(layers)]
    native_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    dtype = torch.bfloat16 if native_bf16 else torch.float16
    model = AutoModel.from_pretrained(model_dir, config=cfg, dtype=dtype, device_map='auto').eval()
    if hasattr(model, 'language_model'):
        model = model.language_model
    first = model.get_input_embeddings().weight.device
    H = tcfg.hidden_size
    lens = np.array([len(x) for x in tok(texts, add_special_tokens=False)['input_ids']])
    order = np.argsort(-lens)
    out = np.zeros((len(layers), len(texts), H), dtype=np.float32)

    def run_rows(rows, dt):
        res = np.zeros((len(layers), len(rows), H), dtype=np.float32)
        for b in range(0, len(rows), batch):
            idx = rows[b:b + batch]
            enc = tok([texts[i] for i in idx], return_tensors='pt', padding=True, truncation=True,
                      max_length=max_len, add_special_tokens=False).to(first)
            with torch.inference_mode(), torch.autocast('cuda', dtype=dt, enabled=dt != torch.float32):
                hs = model(**enc, output_hidden_states=True).hidden_states
            m = enc['attention_mask'].unsqueeze(-1).float(); m[:, 0] = 0
            for j, L in enumerate(layers):
                h = hs[L].float().to(m.device)
                res[j, b:b + len(idx)] = ((h * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
        return res

    out[:, order] = run_rows(order, dtype)
    bad = np.where(~np.isfinite(out).all(axis=(0, 2)))[0]
    if len(bad):
        log(f'{len(bad)} non-finite rows -> recompute with bf16 autocast')
        out[:, bad] = run_rows(bad, torch.bfloat16)
        bad2 = np.where(~np.isfinite(out).all(axis=(0, 2)))[0]
        if len(bad2):
            log(f'{len(bad2)} rows still non-finite -> zeroed')
            out[:, bad2] = 0
    del model; gc.collect(); torch.cuda.empty_cache()
    return out


def centre_chunks(idx, size, rng):
    lab = np.empty(len(idx), dtype=object)
    g = groups_tr[idx]
    for p in np.unique(g):
        mem = rng.permutation(np.where(g == p)[0])
        for c, s in enumerate(range(0, len(mem), size)):
            lab[mem[s:s + size]] = f'{p}_{c}'
    return lab


def centre(M, lab):
    M = M.copy()
    for c in np.unique(lab):
        i = lab == c
        if i.sum() > 1:
            M[i] -= M[i].mean(0)
    return M

### Potential score improvement — frozen hidden-state probes

The frozen-probe branch is a high-value tuning area because it changes representation quality without expensive end-to-end fine-tuning.

Promising experiments:

- **Layer selection search:** adjacent transformer layers often contain redundant information. Search for a smaller but more complementary subset.
- **Pooling variants:** compare mean pooling, last-token pooling, attention-weighted pooling, and mean+std pooling.
- **Longer context:** if GPU memory allows it, test whether increasing `max_len` beyond 512 improves attribution on long responses.
- **Multi-layer concatenation:** instead of averaging per-layer classifiers, concatenate a few selected layer embeddings and train one regularized head.
- **Whitening / PCA:** reduce correlated dimensions before logistic regression.
- **Per-model temperature calibration:** frozen probes may be differently calibrated even when accuracy is similar.

> **Potential score improvement:** a carefully selected multi-layer representation plus calibration is likely a stronger candidate than simply adding more probe layers.

In [ ]:
# Hidden-state probe design space
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(13, 4.2))
ax.set_xlim(0, 13)
ax.set_ylim(0, 4)
ax.axis("off")

steps = [
    (0.3, "Tokens", "truncated input"),
    (2.5, "Selected layers", "early / mid / late"),
    (5.0, "Pooling", "mean / last / mean+std"),
    (7.5, "Projection", "scaler / PCA / whitening"),
    (10.0, "Classifier", "regularized LR"),
    (11.8, "Calibration", "temperature")
]

for x, title, sub in steps:
    w = 1.6 if x < 11 else 1.0
    p = FancyBboxPatch((x, 1.35), w, 1.1,
                       boxstyle="round,pad=0.03",
                       facecolor="white", linewidth=1.3)
    ax.add_patch(p)
    ax.text(x+w/2, 1.95, title, ha="center", va="center",
            fontsize=9.5, fontweight="bold")
    ax.text(x+w/2, 1.58, sub, ha="center", va="center", fontsize=7.8)

for x1, x2 in [(1.9,2.45),(4.1,4.95),(6.6,7.45),(9.1,9.95),(11.6,11.75)]:
    ax.add_patch(FancyArrowPatch((x1,1.9),(x2,1.9),
                                arrowstyle="-|>", mutation_scale=12))

ax.set_title("Frozen-Probe Improvement Search Space", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Recover topic groups from text

Qwen3-Embedding-0.6B receives an instruction to identify the passage's topic or question. Cosine similarity above 0.92 links passages; connected components larger than six are split into singletons. Validation prompt IDs are never supplied to this predictor.

### Topic-group recovery: the structural bridge

The hidden test set does not directly provide the training `prompt_id`, so the notebook reconstructs likely prompt groups from semantic similarity.

The procedure is:

1. Embed every test passage with a **topic-focused instruction**.
2. Compute cosine similarities.
3. Add an edge when similarity exceeds `0.92`.
4. Take connected components.
5. Convert components larger than six into singletons.

This recovered grouping is used in two places:

- to construct **group-centred features**;
- to activate the **exact distinct-author decoder**.

Therefore, grouping errors can propagate into the final prediction. The code deliberately skips oversized groups to limit that risk.


In [ ]:
# Conceptual graph for topic grouping
import numpy as np
import matplotlib.pyplot as plt

pts = np.array([
    [0.0, 0.0], [0.8, 0.3], [0.3, 0.9],     # group A
    [3.0, 0.2], [3.7, 0.7],                  # group B
    [5.4, 1.0],                               # singleton
])
edges = [(0,1),(0,2),(1,2),(3,4)]

fig, ax = plt.subplots(figsize=(9, 3.7))
for i, j in edges:
    ax.plot([pts[i,0], pts[j,0]], [pts[i,1], pts[j,1]], linewidth=1.5)

ax.scatter(pts[:,0], pts[:,1], s=500)
for i, (x,y) in enumerate(pts):
    ax.text(x, y, f"P{i+1}", ha="center", va="center", fontsize=10)

ax.text(0.45, -0.45, "Recovered group A", ha="center")
ax.text(3.35, -0.45, "Recovered group B", ha="center")
ax.text(5.4, 0.55, "singleton", ha="center")
ax.set_title("Cosine-similarity graph → connected components")
ax.set_xlim(-0.8, 6.2)
ax.set_ylim(-0.8, 1.8)
ax.axis("off")
plt.show()


In [ ]:
# ------------------------------------------------------------------ test-time topic groups
import traceback
test_groups = None


def make_groups():
    from sentence_transformers import SentenceTransformer
    from scipy.sparse import csr_matrix
    from scipy.sparse.csgraph import connected_components
    d = find_model_dir(dict(model_type='qwen3', hidden_size=1024, num_hidden_layers=28), LOCAL_MODELS['emb'])
    if d is None:
        raise FileNotFoundError('Qwen3-Embedding-0.6B not found')
    emb = SentenceTransformer(d, device='cuda')
    Qe = emb.encode(te_texts, batch_size=32, normalize_embeddings=True, show_progress_bar=False,
                    prompt='Instruct: Identify the specific topic or question this passage answers\nQuery: ')
    del emb; gc.collect()
    S = Qe @ Qe.T; np.fill_diagonal(S, 0)
    n_c, lab = connected_components(csr_matrix(S > GROUP_THR), directed=False)
    sizes = np.bincount(lab)
    big = sizes[lab] > GROUP_CAP
    lab = lab.copy(); lab[big] = n_c + np.arange(big.sum())  # oversized components -> singletons
    _, lab = np.unique(lab, return_inverse=True)
    sz = np.bincount(lab)[lab]
    log(f'groups: {np.mean(sz > 1):.3f} of test passages grouped; size hist {np.bincount(np.bincount(lab))[1:8].tolist()}')
    return lab


if USE_GROUPS and not args.skip_gpu:
    try:
        test_groups = make_groups()
    except Exception:
        log('!! grouping failed -> continuing without groups\n' + traceback.format_exc())

### Potential score improvement — topic-group recovery

This is probably the most sensitive structural part of the notebook.

The current rule is essentially:

\[
\text{same group} \iff \cos(e_i,e_j) > 0.92
\]

followed by connected components. A single false-positive edge can merge two otherwise correct groups.

High-value experiments:

- **Mutual-nearest-neighbour linking:** require both passages to select each other as strong neighbours.
- **Two-threshold graph:** use a high threshold for seed edges and a lower threshold only for expansion.
- **Cluster confidence:** run exact decoding only when the minimum within-group similarity is high.
- **Prompt-summary embeddings:** first compress each passage into a short inferred question/topic representation, then embed that summary.
- **Alternative embedding models:** compare retrieval-oriented and general semantic encoders.
- **Graph pruning:** remove weak bridge edges that connect two dense sub-clusters.

> **Potential score improvement:** increasing **precision** of recovered groups is more important than increasing recall. Missing a group merely loses a possible gain; a false group can actively distort several predictions.

In [ ]:
# Why one weak bridge can be dangerous
import matplotlib.pyplot as plt
import numpy as np

A = np.array([[0.4,1.0],[1.0,1.4],[1.2,0.6]])
B = np.array([[4.0,1.0],[4.6,1.3],[4.5,0.5]])

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(A[:,0], A[:,1], s=420)
ax.scatter(B[:,0], B[:,1], s=420)

for pts in [A, B]:
    for i in range(len(pts)):
        for j in range(i+1, len(pts)):
            ax.plot([pts[i,0],pts[j,0]],[pts[i,1],pts[j,1]], linewidth=1.2)

# suspicious bridge
ax.plot([A[1,0], B[0,0]],[A[1,1],B[0,1]], linestyle="--", linewidth=1.8)

ax.text(1.0, 2.0, "True topic A", ha="center", fontweight="bold")
ax.text(4.4, 2.0, "True topic B", ha="center", fontweight="bold")
ax.text(2.7, 1.55, "One weak bridge can merge both\nconnected components",
        ha="center", fontsize=9)

ax.set_title("Failure Mode of Threshold + Connected Components")
ax.set_xlim(-0.2, 5.2)
ax.set_ylim(0.0, 2.4)
ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Apply the fitted readouts and LoRA

Frozen readouts use ordinary and group-centred feature views. The full-fit Qwen3-8B LoRA checkpoint is used only for final inference, not for evaluating held-out training prompts. An optional compact CE head is retained for the recorded ablation, but its blend weight is **zero** in this version.

### Plain heads vs. group-centred heads

For grouped passages, the pipeline can replace part of a component's ordinary prediction with a prediction from **centred features**.

Conceptually:

\[
h_i^{centered} = h_i - \frac{1}{|G|}\sum_{j\in G} h_j
\]

This removes information shared by all passages in the same recovered topic group and emphasizes **relative stylistic differences** between authors.

The final blend does not discard the plain head. It moves the grouped rows toward the centred prediction by a component-specific amount:

- Qwen3.5 frozen: full move,
- Gemma4 frozen: full move,
- Qwen3-8B LoRA: 50/50 move.


In [ ]:
# ------------------------------------------------------------------ frozen probes
CENTRED = {}
EXTRA_FEATURES = {}
EXTRA_LAYERS = {'q35': [3,5,18], 'g12': [6,12]}
from hashlib import sha256
weights_file = next(Path('/kaggle/input').rglob('ntl_head_weights.npz'))
weights_manifest = json.loads((weights_file.parent/'manifest.json').read_text())
assert sha256((DATA/'train.csv').read_bytes()).hexdigest() == weights_manifest['train_sha256']
assert sha256(weights_file.read_bytes()).hexdigest() == weights_manifest['head_weights_sha256']
PREFIT = np.load(weights_file,allow_pickle=False)
assert np.array_equal(PREFIT['labels'],LABELS)
log(f'Loaded verified full-fit heads: {weights_file}')


def run_probe(pr):
    d = find_model_dir(pr['match'],LOCAL_MODELS.get(pr['name']))
    if d is None: raise FileNotFoundError(pr['name'])
    assert weights_manifest['layers'][pr['name']] == pr['layers']
    log(f"{pr['name']}: inference-only frozen extraction from {d}")
    Ete = extract(d,te_texts,pr['layers'])
    selected = np.concatenate([Ete[pr['layers'].index(layer)] for layer in EXTRA_LAYERS[pr['name']]],axis=1)
    EXTRA_FEATURES[pr['name']] = np.clip(selected,-65500,65500).astype(np.float16).astype(np.float32)
    acc = np.zeros((len(te_texts),K)); accc = np.zeros_like(acc)
    for j in range(len(pr['layers'])):
        prefix = f"{pr['name']}_{j}_plain"
        x = Ete[j].astype(np.float32)
        z = StandardScaler().fit_transform(x) if TEST_STD and len(x)>=50 else apply_scaler(x,PREFIT[prefix+'_mean'],PREFIT[prefix+'_scale'])
        acc += linear_logproba(z,PREFIT[prefix+'_coef'],PREFIT[prefix+'_bias'])
        if test_groups is not None:
            prefix = f"{pr['name']}_{j}_center"
            z = apply_scaler(centre(x,test_groups),PREFIT[prefix+'_mean'],PREFIT[prefix+'_scale'])
            accc += linear_logproba(z,PREFIT[prefix+'_coef'],PREFIT[prefix+'_bias'])
    COMP[pr['name']] = logp_norm(acc/len(pr['layers']))
    if test_groups is not None: CENTRED[pr['name']] = logp_norm(accc/len(pr['layers']))
    log(f"{pr['name']} prefit heads done")


def run_lora(lo):
    from lora_infer import lora_predict
    base = find_model_dir(lo['base_match'], LOCAL_MODELS.get(lo['name']))
    names = lo['ckpts']
    if args.local_fold >= 0:  # local validation must not use models trained on the held-out fold
        names = [os.environ.get('NTL_LOCAL_LORA', f'q8_L10_ml_v_fold{args.local_fold}')]
    cks = [p for n in names for r in CKPT_ROOTS for p in glob.glob(f'{r}/**/{n}', recursive=True)][:len(names)]
    if base is None or not cks:
        log(f'!! lora {lo["name"]} base/ckpt not found ({base}, {cks}); skipping'); return
    if test_groups is None:
        preds = [lora_predict(base, ck, te_texts, log=log) for ck in cks]
        COMP[lo['name']] = logp_norm(np.mean(preds, 0))
    else:  # two-head checkpoints also give group-centred predictions
        res = [lora_predict(base, ck, te_texts, log=log, groups=test_groups) for ck in cks]
        COMP[lo['name']] = logp_norm(np.mean([r[0] for r in res], 0))
        cens = [r[1] for r in res if r[1] is not None]
        if cens:
            CENTRED[lo['name']] = logp_norm(np.mean(cens, 0))
    log(f'{lo["name"]} done ({len(cks)} checkpoints; centred head: {lo["name"] in CENTRED})')

### Why centering can help — and when it can hurt

Suppose several passages answer the same hidden question. Their hidden representations can be decomposed conceptually into:

\[
h_i = \underbrace{t_G}_{\text{shared topic}} +
      \underbrace{s_i}_{\text{author/style}} +
      \underbrace{\epsilon_i}_{\text{noise}}
\]

Subtracting the group mean approximately removes the shared topic component:

\[
h_i - \bar h_G \approx
(s_i-\bar s_G) + (\epsilon_i-\bar\epsilon_G)
\]

This is useful only if the group is valid. If unrelated passages are grouped together, centering removes meaningful information and introduces artificial contrasts.

> **Potential score improvement:** learn or tune `GROUP_WS` from prompt-held-out OOF predictions separately for each component and possibly as a function of group confidence. A fixed 1.0 / 1.0 / 0.5 routing rule is simple, but confidence-adaptive routing could be stronger.

## 7. Compute component predictions

Missing required components raise an error rather than silently producing a weaker submission. The optional CE projection is computed for inspection; the next cell uses the exact-only blend.

In [ ]:
if not args.skip_gpu:
    for pr in PROBES:
        try:
            run_probe(pr)
        except Exception:
            log(f'!! probe {pr["name"]} failed\n' + traceback.format_exc())
        gc.collect()
        torch.cuda.empty_cache()
    for lo in LORAS:
        try:
            run_lora(lo)
        except Exception:
            log(f'!! lora {lo["name"]} failed\n' + traceback.format_exc())

# Fixed CE projection selected before the independent holdout.
if set(EXTRA_FEATURES) != {'q35','g12'}:
    raise RuntimeError('Missing frozen features for the required CE projection')
X_extra = np.concatenate([EXTRA_FEATURES['q35'],EXTRA_FEATURES['g12']],axis=1)
COMP['ce_mlp'] = prefit_ce_predict(X_extra,PREFIT)
del X_extra,EXTRA_FEATURES
gc.collect()

### Potential score improvement — LoRA and neural-head stage

The LoRA branch currently provides the largest blend weight, so even modest improvements here can matter.

Potential experiments:

- **Probability calibration** of the LoRA output before blending.
- **Layer-pooling ablation** for the task head.
- **Rank / alpha sweep** for LoRA if retraining is possible.
- **Class-balanced or focal-style objectives** if some author confusions dominate.
- **Hard-negative training:** emphasize pairs of authors that the current model repeatedly confuses.
- **Checkpoint ensembling:** average predictions from a few genuinely diverse checkpoints rather than relying on one fine-tuned state.

> **Potential score improvement:** hard-negative training and calibration are attractive because they directly target the confusion structure seen in validation rather than merely increasing model size.

## 8. Blend, decode, and write the submission

Keep the original component weights and centred-head routing, normalize their log-probability blend, then apply exact within-group marginals. Check unique IDs, row count, valid labels, and missing values before writing the Parquet file.

### Final decision logic, step by step

The last stage can be read as a four-step transformation:

**A. Weighted evidence accumulation**

\[
z = \sum_m w_m \log p_m
\]

Each ensemble component contributes log-probability evidence.

**B. Group-centred correction**

Only passages belonging to recovered multi-row groups are adjusted toward their centred-head predictions.

**C. Exact conditional decoding**

The blended probabilities are conditioned on the assumption that authors are distinct inside each recovered group.

**D. Row-wise final label**

The author with the highest final marginal probability is selected independently for each row.

Because step C returns marginals rather than a hard permutation, the final row-wise argmax can still repeat an author even though the probability calculation itself was conditioned on injective assignments.


In [ ]:
# Final-stage logic diagram
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.set_xlim(0, 13)
ax.set_ylim(0, 3)
ax.axis("off")

labels = [
    ("Component\nlog-probs", "6 branches"),
    ("Weighted\nblend", "Σ w·log p"),
    ("Centred-head\ncorrection", "grouped rows"),
    ("Exact marginal\ndecoder", "distinct-author prior"),
    ("argmax", "final author"),
]
xs = [0.3, 2.8, 5.3, 7.9, 10.8]
ws = [1.8, 1.8, 2.0, 2.1, 1.5]

for x, w, (t, s) in zip(xs, ws, labels):
    p = FancyBboxPatch((x, 1.0), w, 1.0,
                       boxstyle="round,pad=0.03",
                       facecolor="white", linewidth=1.4)
    ax.add_patch(p)
    ax.text(x+w/2, 1.62, t, ha="center", va="center",
            fontsize=10, fontweight="bold")
    ax.text(x+w/2, 1.27, s, ha="center", va="center", fontsize=8)

for i in range(len(xs)-1):
    ax.add_patch(FancyArrowPatch(
        (xs[i]+ws[i], 1.5), (xs[i+1]-0.1, 1.5),
        arrowstyle="-|>", mutation_scale=14
    ))

ax.text(6.5, 2.55, "Final inference transformation", ha="center",
        fontsize=14, fontweight="bold")
plt.show()


In [ ]:
# ------------------------------------------------------------------ blend
missing = set(WEIGHTS) - set(COMP)
if missing:
    raise RuntimeError(f'Required ensemble components failed: {sorted(missing)}')
if USE_GROUPS and test_groups is None:
    raise RuntimeError('Required topic grouping failed; refusing silent degraded submission')
used = {k: v for k, v in WEIGHTS.items() if k in COMP}
log(f'blending {used}')
z = sum(w * logp_norm(COMP[k]) for k, w in used.items())
if test_groups is not None and CENTRED:
    grouped = np.bincount(test_groups)[test_groups] > 1
    # for grouped passages, move each component towards its group-centred prediction by GROUP_WS[k]
    for k in CENTRED:
        if k in used:
            gw = GROUP_WS.get(k, GROUP_W)
            z[grouped] += used[k] * gw * (CENTRED[k][grouped] - logp_norm(COMP[k])[grouped])
    log(f'grouped passages: {grouped.mean():.3f}; centred components {list(CENTRED)}')
previous_pred = repulsion(z, test_groups, alpha=.2).argmax(1) if test_groups is not None else z.argmax(1)
z = z / sum(used.values())
if test_groups is not None:
    z = decode(z, test_groups, strength=1., max_group=6)
changed = int(np.sum(previous_pred != z.argmax(1)))
log(f'Exact-only conditional marginal decoder: {changed}/{len(z)} predictions changed relative to original ensemble')
pred = z.argmax(1)
np.savez_compressed(OUT / 'components.npz', **COMP, **{f'centred_{k}': v for k, v in CENTRED.items()})
if y_hold is not None:
    from sklearn.metrics import balanced_accuracy_score as bas
    for k, v in COMP.items():
        log(f'holdout {k}: bacc {bas(y_hold, v.argmax(1)):.4f}')
    log(f'holdout BLEND: bacc {bas(y_hold, pred):.4f}')
sub = pd.DataFrame({'id': test.id.to_numpy(), 'model': LABELS[pred]})
assert sub.id.is_unique and sub.model.isin(LABELS).all() and len(sub) == len(test) and sub.notna().all().all()
sub.to_parquet(OUT / 'submission.parquet', index=False)
log(f'wrote {OUT / "submission.parquet"} ({len(sub)} rows); prediction histogram: {np.bincount(pred, minlength=K).tolist()}')

## Score-improvement map

The table below separates **likely useful experiments** from changes that are elegant but higher-risk.

| Priority | Experiment | Why it may help | Main risk |
|---:|---|---|---|
| 1 | Re-fit blend weights on strict prompt-held-out OOF | directly optimizes final ensemble | validation overfitting |
| 2 | Calibrate component probabilities | log blending assumes meaningful probability scales | calibration set leakage |
| 3 | Improve topic-group precision | protects group correction + exact decoding | lower group recall |
| 4 | Confidence-gated exact decoding | skip uncertain groups instead of forcing structure | threshold sensitivity |
| 5 | Tune `GROUP_WS` by component/confidence | better balance plain vs centred evidence | more hyperparameters |
| 6 | Frozen-probe layer/pooling search | improves latent style features cheaply | GPU/runtime cost |
| 7 | LoRA hard-negative refinement | attacks author confusion directly | retraining cost |
| 8 | TF-IDF/stylometry refinements | adds complementary weak signals | diminishing returns |
| 9 | Larger exact-decoding groups | uses more structure | factorial compute + error amplification |

### Recommended next experiment

A strong next version would keep the exact decoder unchanged and test a **confidence-gated structural layer**:

1. compute group confidence from pairwise embedding similarities;
2. apply group-centred heads only above threshold \(T_1\);
3. apply exact decoding only above a stricter threshold \(T_2\);
4. tune \(T_1,T_2\) exclusively on held-out prompts.

This directly addresses the notebook's main observed pattern: local validation suggests structural information can help, but the public score shows that the current transfer is not yet robust enough.

In [ ]:
# Experiment priority matrix
import matplotlib.pyplot as plt

experiments = [
    "Blend weight refit",
    "Calibration",
    "Group precision",
    "Confidence gating",
    "GROUP_WS tuning",
    "Probe pooling/layers",
    "LoRA hard negatives",
    "Surface features",
    "Larger groups"
]
expected_gain = [4.5, 4.2, 4.5, 4.3, 3.8, 3.9, 4.0, 2.8, 2.3]
risk =          [2.2, 1.8, 2.4, 1.9, 2.7, 2.6, 3.6, 1.8, 4.7]

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(risk, expected_gain, s=[520,480,560,520,450,470,500,400,430])

for x, y, name in zip(risk, expected_gain, experiments):
    ax.text(x + 0.05, y + 0.03, name, fontsize=8.7)

ax.set_xlabel("Implementation / validation risk")
ax.set_ylabel("Expected improvement potential")
ax.set_title("Suggested Experiment Priorities (Conceptual)")
ax.set_xlim(1.4, 5.1)
ax.set_ylim(2.0, 4.9)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Suggested ablation table for the next Kaggle version

For a credible score-improvement story, record both local validation and leaderboard results:

| Version | Base ensemble | Group-centred heads | Exact decoder | Confidence gate | Calibration | Local CV | Public LB |
|---|---:|---:|---:|---:|---:|---:|---:|
| A | ✓ |  |  |  |  |  |  |
| B | ✓ | ✓ |  |  |  |  |  |
| C | ✓ |  | ✓ |  |  |  |  |
| D | ✓ | ✓ | ✓ |  |  |  |  |
| E | ✓ | ✓ | ✓ | ✓ |  |  |  |
| F | ✓ | ✓ | ✓ | ✓ | ✓ |  |  |

This makes it much easier to determine whether gains come from better probabilities, better structural assumptions, or simply validation noise.

## What the experiments showed

All development splits held out entire prompts. Of 150 prompts, 30 were reserved before model selection; the other 120 supplied four development folds. Scalers and classifiers were fitted within each training fold. Validation pools simulated one, three, six, mixed numbers of responses, and unequal author frequencies, with 12 deterministic samples per regime.

**Frozen/CPU development results, excluding the published full-fit LoRA:**

| Method | Mean balanced accuracy across five regimes |
|---|---:|
| Original frozen/CPU control | 0.48047 |
| Exact marginal decoding | 0.48665 |
| Compact CE head (15%) + exact decoding | 0.49173 |

The prompt-bootstrap 95% interval for exact-only development gain was [0.00400, 0.00830]. These figures are **not leaderboard scores** and do not validate the complete LoRA ensemble.

The selected CE+exact candidate was then checked once on the reserved 30 prompts: mean 0.54728 → 0.55610, with a prompt-bootstrap gain interval of [−0.00475, +0.02273]. The interval includes zero, and the mixed-size regime worsened.

**Actual Kaggle public results:**

| Full inference pipeline | Public score |
|---|---:|
| Referenced original pipeline | **0.522** |
| CE head 15% + exact decoder, version 4 | 0.518 |
| Exact decoder only, version 5 | **0.520** |

The exact-only ablation was declared before obtaining the first leaderboard result. Local gains did not transfer to the full ensemble. Possible explanations include the missing LoRA branch in local validation, calibration differences, and a different context distribution in the hidden test; these are hypotheses, not established causes.

Other development experiments—complete-link grouping, partial topic projection, partial test-statistics adaptation, a contrastive projection, and a filtered external corpus—were not selected.

## Credits and reproducibility

- [Ahmed Berat Ozer, `name-that-llm-d` v1](https://www.kaggle.com/code/ahmedberatozer/name-that-llm-d?scriptVersionId=358776684): base inference, frozen readouts, LoRA checkpoint, text/style branches and semantic grouping. Apache-2.0.
- Public Qwen and Gemma backbones and the attached input datasets retain their respective licenses.
- This experiment adds the exact marginal decoder, prompt-safe development/holdout evaluation, and the compact-head ablation.

Train SHA256: `40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707`.

The prediction code below the explanations is identical, by Python AST, to the saved version-5 code that produced 0.520. The notebook does not claim a new best score or verified OOF performance for the complete LoRA ensemble.

### Interpreting the reported results carefully

The reported local improvement from the exact decoder is real **within the stated validation design**, but the complete public pipeline did not improve over the referenced baseline.

That gap suggests at least one of the following:

- recovered test groups are noisier than validation prompt groups;
- model calibration differs between local OOF and full-fit inference;
- the LoRA branch changes the error geometry of the ensemble;
- the structural assumption is only beneficial on a subset of groups;
- the observed local gain is too small relative to leaderboard noise / distribution shift.

Therefore, the most defensible conclusion is not “the decoder improves the score,” but:

> **The decoder is conditionally useful when group recovery is reliable. The next optimization target should be deciding *when* to trust the structural prior.**

---

# Practical interpretation

This notebook is best understood as a **hybrid attribution system**:

- classical text statistics detect surface habits;
- frozen LLMs provide rich latent style representations;
- LoRA adds a task-specialized neural signal;
- semantic embeddings reconstruct hidden prompt structure;
- centred heads suppress common topic information;
- exact marginal decoding injects a local structural prior.

## What is experimentally important?

The notebook's own results show that the elegant group-level decoder **improved local frozen/CPU validation but did not improve the complete public leaderboard pipeline**. That makes this notebook especially useful as an example of the difference between:

- a theoretically well-motivated structural constraint,
- a local validation gain,
- and actual transfer to the hidden leaderboard distribution.

## Main failure modes to keep in mind

1. **Incorrect topic grouping** can force unrelated passages to interact.
2. **Calibration mismatch** between ensemble components changes how strongly the exact decoder redistributes probability.
3. The **LoRA branch was not fully represented in the local validation experiment**, so gains measured locally may not transfer to the final ensemble.
4. The structural assumption “one author per recovered prompt group” may not perfectly match hidden-test construction.
5. Test-statistics adaptation can change the geometry of frozen features when the test distribution differs from training.

## Bottom line

The core novelty is not a new backbone. It is a **probabilistic decoding layer placed on top of an already strong multi-model ensemble**. The notebook is therefore a good reference for structural post-processing, group-aware classification, and the risks of transferring local gains to a leaderboard.
